# Phishing URL Detector
**Author:** YourName  
**Program:** IBM SkillsBuild Cyber Security 2026 (BharatCares)

A simple machine learning project that classifies URLs as *legitimate* or *phishing* using features extracted from the URL text.

In [ ]:
import re
import pandas as pd
from urllib.parse import urlparse
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

## 1. Dataset
A small labelled sample is built in so the notebook runs anywhere. 1 = phishing, 0 = legitimate. For a bigger experiment, use the UCI Phishing Websites dataset linked in the README.

In [ ]:
legit = [
 "https://www.google.com/search?q=python","https://github.com/scikit-learn/scikit-learn",
 "https://www.wikipedia.org/wiki/Cybersecurity","https://www.ibm.com/skills","https://www.amazon.in/dp/B08N5WRWNW",
 "https://docs.python.org/3/library/re.html","https://www.microsoft.com/en-in/windows","https://stackoverflow.com/questions/11227809",
 "https://www.linkedin.com/in/someone","https://www.bbc.com/news","https://www.irctc.co.in/nget/train-search",
 "https://www.hdfcbank.com/personal/pay/cards","https://www.flipkart.com/mobiles","https://www.youtube.com/watch?v=dQw4w9WgXcQ",
 "https://mail.google.com/mail/u/0/","https://www.sbi.co.in/web/personal-banking","https://www.apple.com/in/iphone",
 "https://www.coursera.org/learn/machine-learning","https://www.nptel.ac.in/courses","https://www.paypal.com/in/home",
]
phish = [
 "http://secure-paypal-login.verify-account.com/signin","http://192.168.45.10/bank/login.php",
 "http://google.com.account-update.xyz/verify?id=839201","http://amazon-security-alert.tk/confirm-payment",
 "http://sbi-online-kyc-update.info/login@secure","http://paypa1.com-webscr.net/cgi-bin/update",
 "http://bit.ly-free-gift-card.top/claim?user=abc","http://hdfc-netbanking-verify.co/auth/login.html",
 "http://microsoft-support-alert.win/unlock-account","http://login-facebook.com.security-check.ml/home",
 "http://45.77.12.9/wp-admin/paytm/login","http://apple-id-locked.verify-now.cf/restore",
 "http://netflix-billing-update.click/payment","http://irctc-refund-claim.xyz/login?session=1",
 "http://secure.bankofindia.verify-login.online/@update","http://free-recharge-offer.gq/win-now",
 "http://instagram-copyright-appeal.tk/form","http://dropbox.com.shared-doc.review/open",
 "http://upi-cashback-reward.buzz/collect","http://whatsapp-gold-update.top/download.apk",
]
df = pd.DataFrame({"url": legit+phish, "label": [0]*len(legit)+[1]*len(phish)})
df.head()

## 2. Feature extraction
Phishing URLs tend to be long, use raw IP addresses, contain `@`, many hyphens/dots, no HTTPS, and suspicious keywords or TLDs.

In [ ]:
SUSPICIOUS_WORDS = ["login","verify","secure","update","account","confirm","payment","bank","free","claim","alert","kyc","unlock"]
SUSPICIOUS_TLDS = (".tk",".ml",".ga",".cf",".gq",".xyz",".top",".click",".buzz",".win",".online",".review",".info")

def extract_features(url):
    parsed = urlparse(url)
    host = parsed.netloc.lower()
    return {
        "url_length": len(url),
        "host_length": len(host),
        "num_dots": host.count("."),
        "num_hyphens": host.count("-"),
        "has_at": int("@" in url),
        "has_ip": int(bool(re.fullmatch(r"\d{1,3}(\.\d{1,3}){3}", host))),
        "uses_https": int(parsed.scheme == "https"),
        "num_digits": sum(c.isdigit() for c in url),
        "num_keywords": sum(w in url.lower() for w in SUSPICIOUS_WORDS),
        "bad_tld": int(host.endswith(SUSPICIOUS_TLDS)),
        "num_subdomains": max(host.count(".") - 1, 0),
    }

X = pd.DataFrame([extract_features(u) for u in df.url])
y = df.label
X.head()

## 3. Train and evaluate

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)
pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, pred))
print(confusion_matrix(y_test, pred))
print(classification_report(y_test, pred, target_names=["legitimate","phishing"]))

In [ ]:
pd.Series(model.feature_importances_, index=X.columns).sort_values(ascending=False)

## 4. Try your own URL

In [ ]:
def check_url(url):
    feats = pd.DataFrame([extract_features(url)])
    p = model.predict_proba(feats)[0][1]
    verdict = "PHISHING" if p >= 0.5 else "LEGITIMATE"
    print(f"{url}\n  -> {verdict} (phishing probability {p:.2f})")

check_url("https://www.google.com")
check_url("http://secure-bank-login.verify-account.tk/update")

## 5. Conclusion
Simple URL-based features are enough to flag many phishing links, but a larger dataset and extra features (domain age, WHOIS, page content) would make the detector more reliable.